In [1]:
!pip install xgboost shap langchain transformers torch fastapi uvicorn langchain-community


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
print(sys.executable)

import numpy, sklearn, xgboost
print(numpy.__version__)
print("imports ok")


C:\Users\Aung Thu Hein\anaconda3\python.exe
1.26.4
imports ok


In [4]:
import xgboost as xgb
import pickle

model = pickle.load(open("./model/xgb_model.pkl", "rb"))

In [5]:
def predict_risk(data):
    import numpy as np

    data = np.array(data)          # convert to numpy
    print("Before reshape:", data.shape)

    data = data.reshape(1, -1)     # ✅ MUST DO
    print("After reshape:", data.shape)

    proba = model.predict_proba(data)[0][1]

    if proba < 0.3:
        risk = "Low"
    elif proba < 0.7:
        risk = "Medium"
    else:
        risk = "High"

    return proba, risk

In [6]:
import shap

explainer = shap.Explainer(model)

def get_top_features(data, feature_names):
    import numpy as np
    import shap

    data = np.array(data).reshape(1, -1)  # ✅ MUST

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(data)[0]

    feature_impact = list(zip(feature_names, shap_values))
    feature_impact.sort(key=lambda x: abs(x[1]), reverse=True)

    top_features = feature_impact[:3]

    result = []
    for f, v in top_features:
        impact = "High" if abs(v) > 0.3 else "Medium"
        result.append({"feature": f, "impact": impact})

    return result

In [9]:
pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.


In [24]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# hg_pipeline = pipeline(
#     "text-generation",
#     model="google/flan-t5-base",
#     max_new_tokens=200,
#     device_map="auto",
# )

# hg_pipeline = pipeline(
#     "text2text-generation",
#     model="google/flan-t5-base",
#     max_new_tokens=200,
#     device_map="auto"
# )

# llm = HuggingFacePipeline(pipeline=hg_pipeline)

from langchain_community.llms.ollama import Ollama

llm = Ollama(model="qwen3:8b")

In [16]:
from langchain_core.prompts import PromptTemplate

template = """
You are a health assistant.

Risk Level: {risk}
Probability: {probability}

Top contributing factors:
{factors}

Provide:
1. Simple explanation
2. Lifestyle advice
3. Preventive steps
4. Suggest consulting a doctor

Do NOT give medical diagnosis.
"""

prompt = PromptTemplate(
    input_variables=["risk", "probability", "factors"],
    template=template,
)

In [17]:
def generate_advice(risk, probability, factors):
  formatted_factors = "\n".join(
          [f"- {f['feature']}: {f['impact']}" for f in factors]
      )

  final_prompt = prompt.format(
        risk=risk,
        probability=round(probability, 2),
        factors=formatted_factors
    )

  response = llm.invoke(final_prompt)
  return response

In [18]:
def prepare_input(data):
    # unpack original 8 features
    (Pregnancies, Glucose, BloodPressure,
     SkinThickness, Insulin, BMI,
     DiabetesPedigreeFunction, Age) = data

    # create missing flags
    Missing_Insulin = 1 if Insulin == 0 else 0
    Missing_SkinThickness = 1 if SkinThickness == 0 else 0

    # return full 10 features
    return [
        Pregnancies, Glucose, BloodPressure,
        SkinThickness, Insulin, BMI,
        DiabetesPedigreeFunction, Age,
        Missing_Insulin, Missing_SkinThickness
    ]

In [19]:
def analyze_risk(input_data):
    feature_names = [
        "Pregnancies", "Glucose", "BloodPressure",
        "SkinThickness", "Insulin", "BMI",
        "DiabetesPedigreeFunction", "Age",
        "Missing_Insulin", "Missing_SkinThickness"
    ]

    # ✅ FIX: prepare input
    full_input = prepare_input(input_data)

    probability, risk = predict_risk(full_input)
    print(probability, risk)
    factors = get_top_features(full_input, feature_names)
    print(factors)
    advice = generate_advice(risk, probability, factors)

    return {
        "risk": risk,
        "probability": probability,
        "top_factors": factors,
        "advice": advice
    }

In [25]:
print(analyze_risk([2, 180, 85, 30, 0, 32.5, 0.5, 45]))

Before reshape: (10,)
After reshape: (1, 10)
0.7171334 High
[{'feature': 'Glucose', 'impact': 'High'}, {'feature': 'Insulin', 'impact': 'High'}, {'feature': 'Age', 'impact': 'High'}]
{'risk': 'High', 'probability': 0.7171334, 'top_factors': [{'feature': 'Glucose', 'impact': 'High'}, {'feature': 'Insulin', 'impact': 'High'}, {'feature': 'Age', 'impact': 'High'}], 'advice': '1. **Simple Explanation**  \nYour risk level is high due to elevated glucose (blood sugar), insulin levels, and age. These factors are linked to conditions like diabetes, metabolic syndrome, or heart disease. While this doesn’t mean you have a specific illness, it highlights the need to address these risks early to protect your health.\n\n2. **Lifestyle Advice**  \n- **Diet**: Focus on whole foods (vegetables, lean proteins, whole grains) and limit refined sugars, processed foods, and sugary drinks.  \n- **Exercise**: Aim for 30–60 minutes of moderate activity (e.g., walking, swimming) most days to improve insulin se